# Eval Results Analysis

Aggregates a user-supplied list of `paper_eval` run directories into a single
table with one row per run and one column per benchmark, plus an `average`
column. No recursive directory sweep is performed.

**Expected layout**:

```
<root>/<model-name>/paper_eval_results/paper_eval_<timestamp>/
    results.json
    eval_resolved_config.yaml
    predictions/
        boolq.json
        piqa.json
        ...
```


In [22]:
import json
from pathlib import Path

import pandas as pd

# Same task order as eval.py
DEFAULT_TASKS = [
    "boolq",
    "piqa",
    "social_i_qa",
    "ARC-Challenge",
    "ARC-Easy",
    "openbookqa",
    "hellaswag",
    "winogrande",
]

ANSWER_PATTERNS = {
    "boolq": r"true|false",
    "piqa": r"solution1|solution2",
    "social_i_qa": r"answer1|answer2|answer3|answer4|answer5",
    "ARC-Challenge": r"answer1|answer2|answer3|answer4|answer5",
    "ARC-Easy": r"answer1|answer2|answer3|answer4|answer5",
    "openbookqa": r"answer1|answer2|answer3|answer4|answer5",
    "hellaswag": r"ending1|ending2|ending3|ending4",
    "winogrande": r"option1|option2",
}

In [23]:
RUNS = {
    "qdora-seed42-fp32":  {"run_dir": Path("/teamspace/lightning_storage/models/qdora-seed42-fp32/paper_eval_results/paper_eval_20260825_015812/"), "label": "QDoRA (seed 42) - fp32"},
    "qdora-seed43-fp32":  {"run_dir": Path("/teamspace/lightning_storage/models/qdora-seed43-fp32/paper_eval_results/paper_eval_20260825_015842/"), "label": "QDoRA (seed 43) - fp32"},
    "lora-v3":            {"run_dir": Path("/teamspace/lightning_storage/models/lora-v3/paper_eval_results/paper_eval_20260827_103908/"), "label": "LoRA (seed 42)"},
    "lora-match-seed42":  {"run_dir": Path("/teamspace/lightning_storage/models/lora-match-seed42/paper_eval_results/paper_eval_20260717_002852/"), "label": "LoRA LR-Match (seed 42)"},
    "lora-match-seed43":  {"run_dir": Path("/teamspace/lightning_storage/models/lora-match-seed43-v2/paper_eval_results/paper_eval_20260820_194824/"), "label": "LoRA LR-Match (seed 43)"},
    "dora-seed42":        {"run_dir": Path("/teamspace/lightning_storage/models/dora/paper_eval_results/paper_eval_20260827_022522/"), "label": "DoRA (seed 42)"},
    "dora-seed43":        {"run_dir": Path("/teamspace/lightning_storage/models/dora-seed43/paper_eval_results/paper_eval_20260722_130811/"), "label": "DoRA (seed 43)"},
    "qlora-seed42":       {"run_dir": Path("/teamspace/lightning_storage/models/qlora/paper_eval_results/paper_eval_20260814_114804/"), "label": "QLoRA (seed 42)"},
    "qlora-seed43":       {"run_dir": Path("/teamspace/lightning_storage/models/qlora-seed43/paper_eval_results/paper_eval_20260721_213142/"), "label": "QLoRA (seed 43)"},
    # "qdora-seed42":       {"run_dir": Path("/teamspace/lightning_storage/models/qdora/paper_eval_results/paper_eval_20260710_201127/"), "label": "QDoRA (seed 42)"},
    # "qdora-seed43":       {"run_dir": Path("/teamspace/lightning_storage/models/qdora-seed43/paper_eval_results/paper_eval_20260721_213155/"), "label": "QDoRA (seed 43)"},
}


In [24]:
def prepare_run_dirs(runs: dict) -> dict:
    """Validate run dirs and resolve them, keyed by run name."""
    resolved = {}
    problems = []

    for key, spec in runs.items():
        run_dir = Path(spec["run_dir"]).expanduser()
        if not run_dir.is_dir():
            problems.append(f"Directory does not exist: {run_dir}")
            continue
        if not run_dir.name.startswith("paper_eval_"):
            problems.append(f"Not a paper_eval run directory: {run_dir}")
            continue
        if not (run_dir / "predictions").is_dir() and not (
            run_dir / "results.json"
        ).is_file():
            problems.append(
                f"Missing both predictions/ and results.json: {run_dir}"
            )
            continue

        resolved[key] = {"run_dir": run_dir.resolve(), "label": spec["label"]}

    if problems:
        details = "\n  - ".join(problems)
        raise ValueError(f"Invalid RUNS entries:\n  - {details}")

    return resolved


runs_resolved = prepare_run_dirs(RUNS)
print(f"Using {len(runs_resolved)} run dir(s):")
for key, spec in runs_resolved.items():
    print(" ", key, "->", spec["run_dir"])


Using 9 run dir(s):
  qdora-seed42-fp32 -> /teamspace/lightning_storage/models/qdora-seed42-fp32/paper_eval_results/paper_eval_20260825_015812
  qdora-seed43-fp32 -> /teamspace/lightning_storage/models/qdora-seed43-fp32/paper_eval_results/paper_eval_20260825_015842
  lora-v3 -> /teamspace/lightning_storage/models/lora-v3/paper_eval_results/paper_eval_20260827_103908
  lora-match-seed42 -> /teamspace/lightning_storage/models/lora-match-seed42/paper_eval_results/paper_eval_20260717_002852
  lora-match-seed43 -> /teamspace/lightning_storage/models/lora-match-seed43-v2/paper_eval_results/paper_eval_20260820_194824
  dora-seed42 -> /teamspace/lightning_storage/models/dora/paper_eval_results/paper_eval_20260827_022522
  dora-seed43 -> /teamspace/lightning_storage/models/dora-seed43/paper_eval_results/paper_eval_20260722_130811
  qlora-seed42 -> /teamspace/lightning_storage/models/qlora/paper_eval_results/paper_eval_20260814_114804
  qlora-seed43 -> /teamspace/lightning_storage/models/qlora-s

## Per-run accuracy


In [25]:
def task_accuracy_from_predictions(pred_path: Path) -> tuple[int, int, int]:
    """Return (correct, total, parse_failures) from a predictions JSON file."""
    with pred_path.open() as f:
        rows = json.load(f)
    correct = sum(1 for r in rows if r.get("correct"))
    parse_failures = sum(1 for r in rows if r.get("pred", "") == "")
    return correct, len(rows), parse_failures


def collect_run(label: str, run_dir: Path) -> dict:
    info = {
        "model": label,
        "run": run_dir.name,
        "run_dir": str(run_dir),
    }
    results_path = run_dir / "results.json"
    task_data = {}

    if results_path.exists():
        with results_path.open() as f:
            payload = json.load(f)
        for task, res in payload.get("results", {}).items():
            task_data[task] = {
                "accuracy": res.get("accuracy"),
                "correct": res.get("correct"),
                "total": res.get("total"),
                "parse_failures": res.get("parse_failures"),
            }

    # Also scan predictions/ — picks up tasks that finished after the last
    # results.json checkpoint, or runs that never wrote a results.json.
    pred_dir = run_dir / "predictions"
    if pred_dir.is_dir():
        for pred_file in pred_dir.glob("*.json"):
            task = pred_file.stem
            if task in task_data and task_data[task]["accuracy"] is not None:
                continue  # trust results.json
            correct, total, parse_fail = task_accuracy_from_predictions(pred_file)
            task_data[task] = {
                "accuracy": correct / total if total else 0.0,
                "correct": correct,
                "total": total,
                "parse_failures": parse_fail,
            }

    info["tasks"] = task_data
    return info


runs = [collect_run(spec["label"], spec["run_dir"]) for spec in runs_resolved.values()]


## Build the results table

One row per run, one column per benchmark, plus `average` = mean of the
available task accuracies for that run

In [26]:
def build_accuracy_table(runs: list[dict], task_order: list[str]) -> pd.DataFrame:
    rows = []
    for run in runs:
        row = {"model": run["model"], "run": run["run"]}
        for task in task_order:
            entry = run["tasks"].get(task)
            row[task] = entry["accuracy"] if entry else float("nan")
        # Include any extra tasks not in the default list
        for task, entry in run["tasks"].items():
            if task not in task_order:
                row[task] = entry["accuracy"]
        row["run_dir"] = run["run_dir"]
        rows.append(row)

    if not rows:
        return pd.DataFrame(
            columns=["model", "run", *task_order, "average", "run_dir"]
        )

    df = pd.DataFrame(rows)
    task_cols = [c for c in df.columns if c not in {"model", "run", "run_dir"}]
    df["average"] = df[task_cols].mean(axis=1, skipna=True)
    # Column order: identifiers, tasks (in DEFAULT_TASKS order first), average, run_dir
    ordered = (
        ["model", "run"]
        + [c for c in task_order if c in df.columns]
        + [c for c in task_cols if c not in task_order]
        + ["average", "run_dir"]
    )
    return df[ordered]


results_df = build_accuracy_table(runs, DEFAULT_TASKS)

results_df.reset_index(drop=True, inplace=True)
results_df

,model,run,boolq,piqa,social_i_qa,ARC-Challenge,ARC-Easy,openbookqa,hellaswag,winogrande,average,run_dir
0,QDoRA (seed 42) - fp32,paper_eval_20260825_015812,0.759633,0.885745,0.807062,0.797782,0.902778,0.864,0.955288,0.851618,0.852988,/teamspace/lightning_storage/models/qdora-seed...
1,QDoRA (seed 43) - fp32,paper_eval_20260825_015842,0.756881,0.880849,0.805527,0.788396,0.904882,0.862,0.957379,0.864246,0.852520,/teamspace/lightning_storage/models/qdora-seed...
2,LoRA (seed 42),paper_eval_20260827_103908,0.714067,0.849837,0.787615,0.741468,0.852694,0.826,0.922227,0.822415,0.814540,/teamspace/lightning_storage/models/lora-v3/pa...
3,LoRA LR-Match (seed 42),paper_eval_20260717_002852,0.750459,0.880305,0.799386,0.800341,0.908249,0.848,0.953197,0.871350,0.851411,/teamspace/lightning_storage/models/lora-match...
4,LoRA LR-Match (seed 43),paper_eval_20260820_194824,0.751070,0.889010,0.794268,0.780717,0.903620,0.848,0.955387,0.854775,0.847106,/teamspace/lightning_storage/models/lora-match...
5,DoRA (seed 42),paper_eval_20260827_022522,0.756575,0.881393,0.806039,0.796075,0.898148,0.842,0.953894,0.868982,0.850388,/teamspace/lightning_storage/models/dora/paper...
6,DoRA (seed 43),paper_eval_20260722_130811,0.752905,0.883025,0.795803,0.794369,0.907828,0.864,0.954889,0.863457,0.852035,/teamspace/lightning_storage/models/dora-seed4...
7,QLoRA (seed 42),paper_eval_20260814_114804,0.750765,0.883025,0.808598,0.804608,0.907407,0.862,0.954889,0.868193,0.854936,/teamspace/lightning_storage/models/qlora/pape...
8,QLoRA (seed 43),paper_eval_20260721_213142,0.749541,0.885201,0.805527,0.807167,0.909933,0.866,0.955786,0.863457,0.855327,/teamspace/lightning_storage/models/qlora-seed...


## Counts table (correct / total per task)


In [27]:
def build_counts_table(runs: list[dict], task_order: list[str]) -> pd.DataFrame:
    rows = []
    for run in runs:
        row = {"model": run["model"], "run": run["run"]}
        for task in task_order:
            entry = run["tasks"].get(task)
            row[task] = f"{entry['correct']}/{entry['total']}" if entry else ""
        rows.append(row)
    return pd.DataFrame(rows)


counts_df = build_counts_table(runs, DEFAULT_TASKS)
counts_df

,model,run,boolq,piqa,social_i_qa,ARC-Challenge,ARC-Easy,openbookqa,hellaswag,winogrande
0,QDoRA (seed 42) - fp32,paper_eval_20260825_015812,2484/3270,1628/1838,1577/1954,935/1172,2145/2376,432/500,9593/10042,1079/1267
1,QDoRA (seed 43) - fp32,paper_eval_20260825_015842,2475/3270,1619/1838,1574/1954,924/1172,2150/2376,431/500,9614/10042,1095/1267
2,LoRA (seed 42),paper_eval_20260827_103908,2335/3270,1562/1838,1539/1954,869/1172,2026/2376,413/500,9261/10042,1042/1267
3,LoRA LR-Match (seed 42),paper_eval_20260717_002852,2454/3270,1618/1838,1562/1954,938/1172,2158/2376,424/500,9572/10042,1104/1267
4,LoRA LR-Match (seed 43),paper_eval_20260820_194824,2456/3270,1634/1838,1552/1954,915/1172,2147/2376,424/500,9594/10042,1083/1267
5,DoRA (seed 42),paper_eval_20260827_022522,2474/3270,1620/1838,1575/1954,933/1172,2134/2376,421/500,9579/10042,1101/1267
6,DoRA (seed 43),paper_eval_20260722_130811,2462/3270,1623/1838,1555/1954,931/1172,2157/2376,432/500,9589/10042,1094/1267
7,QLoRA (seed 42),paper_eval_20260814_114804,2455/3270,1623/1838,1580/1954,943/1172,2156/2376,431/500,9589/10042,1100/1267
8,QLoRA (seed 43),paper_eval_20260721_213142,2451/3270,1627/1838,1574/1954,946/1172,2162/2376,433/500,9598/10042,1094/1267


## Parse-failure check

In [28]:
pf_rows = []
for run in runs:
    row = {"model": run["model"], "run": run["run"]}
    for task in DEFAULT_TASKS:
        entry = run["tasks"].get(task)
        if entry and entry["total"]:
            row[task] = entry["parse_failures"] / entry["total"]
        else:
            row[task] = float("nan")
    pf_rows.append(row)

parse_fail_df = pd.DataFrame(pf_rows)
parse_fail_df.style.format(
    {c: "{:.2%}" for c in parse_fail_df.columns if c not in {"model", "run"}}
).map(
    lambda v: "background-color: #ffcccc" if isinstance(v, float) and v > 0.05 else "",
    subset=[c for c in parse_fail_df.columns if c not in {"model", "run"}],
)

,model,run,boolq,piqa,social_i_qa,ARC-Challenge,ARC-Easy,openbookqa,hellaswag,winogrande
0,QDoRA (seed 42) - fp32,paper_eval_20260825_015812,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
1,QDoRA (seed 43) - fp32,paper_eval_20260825_015842,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
2,LoRA (seed 42),paper_eval_20260827_103908,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
3,LoRA LR-Match (seed 42),paper_eval_20260717_002852,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
4,LoRA LR-Match (seed 43),paper_eval_20260820_194824,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
5,DoRA (seed 42),paper_eval_20260827_022522,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
6,DoRA (seed 43),paper_eval_20260722_130811,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
7,QLoRA (seed 42),paper_eval_20260814_114804,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%
8,QLoRA (seed 43),paper_eval_20260721_213142,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%,0.00%


## Export

In [29]:
OUT_CSV = Path("outputs/eval_results_table.csv")
results_df.to_csv(OUT_CSV, index=False)
print(f"Wrote {OUT_CSV.resolve()}")


Wrote /teamspace/studios/this_studio/dora-case-studies/notebooks/outputs/eval_results_table.csv


## Drill into a specific run


In [30]:
# preds_df.columns

In [31]:
RUN_INDEX = 0  # row in results_df
TASK = "boolq"

if not results_df.empty:
    run_dir = Path(results_df.iloc[RUN_INDEX]["run_dir"])
    pred_path = run_dir / "predictions" / f"{TASK}.json"
    if pred_path.exists():
        with pred_path.open() as f:
            preds = json.load(f)
        preds_df = pd.DataFrame(preds)
        cols = [c for c in ["gold", "pred", "correct", "output_pred"] if c in preds_df.columns]
        print(f"{len(preds_df)} examples — first 10:")
        display(preds_df[cols].head(10))

        print("\nParse failures (pred == ''):")
        display(preds_df[preds_df["pred"] == ""][cols].head(10))
    else:
        print(f"No predictions file at {pred_path}")

3270 examples — first 10:


,gold,pred,correct,output_pred
0,false,true,False,the correct answer is true
1,true,false,False,the correct answer is false
2,true,true,True,the correct answer is true
3,true,true,True,the correct answer is true
4,true,true,True,the correct answer is true
5,false,true,False,the correct answer is true
6,true,true,True,the correct answer is true
7,true,true,True,the correct answer is true
8,true,true,True,the correct answer is true
9,true,true,True,the correct answer is true



Parse failures (pred == ''):


,gold,pred,correct,output_pred
